# Proyecto PYE - Analisis UYU/BRL\n\nNotebook preparado para Google Colab. Reproduce la carga, validacion, analisis descriptivo, graficos e inferencia basica del tipo de cambio entre el peso uruguayo y el real brasileno.\n\n**Variable principal:** `uyu_por_brl`, interpretada como pesos uruguayos necesarios para comprar un real brasileno.

## 1. Instalar dependencias\n\nEn Colab, la mayoria de las librerias ya estan instaladas. Esta celda asegura que esten disponibles las necesarias.

In [ ]:
!pip -q install pandas numpy matplotlib scipy statsmodels openpyxl

## 2. Cargar el dataset\n\nOpcion recomendada: leer el CSV desde GitHub. Si el archivo cambia de rama o nombre, modificar `DATA_URL`. Tambien se puede subir manualmente el CSV a Colab y cambiar `pd.read_csv(DATA_URL)` por `pd.read_csv('dataset_uyu_brl_2005_2026.csv')`.

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom scipy import stats\n\nDATA_URL = 'https://raw.githubusercontent.com/valentinzamit26/Proyecto-PYE/main/dataset_uyu_brl_2005_2026.csv'\ndf = pd.read_csv(DATA_URL)\ndf['fecha'] = pd.to_datetime(df['fecha'])\ndf = df.sort_values('fecha').reset_index(drop=True)\n\ndf.head()

## 3. Validaciones basicas\n\nSe valida que la variable principal y su inversa no esten confundidas. Debe cumplirse aproximadamente `uyu_por_brl * brl_por_uyu = 1`.

In [ ]:
expected = ['fecha', 'brl_por_usd', 'uyu_por_usd', 'brl_por_uyu', 'uyu_por_brl', 'retorno_diario_brl_por_uyu', 'log_retorno_brl_por_uyu']\nmissing = [col for col in expected if col not in df.columns]\nif missing:\n    raise ValueError(f'Faltan columnas: {missing}')\n\nif 'retorno_diario_uyu_por_brl' not in df.columns:\n    df['retorno_diario_uyu_por_brl'] = df['uyu_por_brl'].pct_change()\nif 'log_retorno_uyu_por_brl' not in df.columns:\n    df['log_retorno_uyu_por_brl'] = np.log(df['uyu_por_brl']).diff()\n\nvalidation = pd.DataFrame({\n    'indicador': ['filas', 'variables', 'fecha_minima', 'fecha_maxima', 'nulos_totales', 'duplicados_fecha', 'max_error_producto_inverso'],\n    'valor': [len(df), df.shape[1], df['fecha'].min().date(), df['fecha'].max().date(), df.isna().sum().sum(), df['fecha'].duplicated().sum(), (df['uyu_por_brl'] * df['brl_por_uyu'] - 1).abs().max()]\n})\nvalidation

## 4. Estadistica descriptiva\n\nResumen de la variable principal `uyu_por_brl`.

In [ ]:
x = df['uyu_por_brl'].dropna()\nq1 = x.quantile(0.25)\nq3 = x.quantile(0.75)\niqr = q3 - q1\nlower = q1 - 1.5 * iqr\nupper = q3 + 1.5 * iqr\n\ndescriptivo = pd.DataFrame({\n    'medida': ['n', 'media', 'mediana', 'q1', 'q3', 'iqr', 'varianza_muestral', 'desvio_estandar_muestral', 'minimo', 'maximo', 'rango', 'coeficiente_variacion', 'asimetria', 'curtosis', 'outliers_iqr'],\n    'valor': [len(x), x.mean(), x.median(), q1, q3, iqr, x.var(ddof=1), x.std(ddof=1), x.min(), x.max(), x.max() - x.min(), x.std(ddof=1) / x.mean(), x.skew(), x.kurt(), ((x < lower) | (x > upper)).sum()]\n})\ndescriptivo

## 5. Graficos principales\n\nEstos graficos son los candidatos principales para el informe. No conviene insertar todos si el PDF supera 13 paginas.

In [ ]:
plt.figure(figsize=(12, 4))\nplt.plot(df['fecha'], df['uyu_por_brl'], linewidth=1)\nplt.title('Evolucion diaria del tipo de cambio UYU/BRL')\nplt.xlabel('Fecha')\nplt.ylabel('UYU por 1 BRL')\nplt.grid(alpha=0.3)\nplt.show()\n\nfig, axes = plt.subplots(1, 2, figsize=(12, 4))\naxes[0].hist(x, bins=40, edgecolor='white')\naxes[0].set_title('Histograma de UYU/BRL')\naxes[0].set_xlabel('UYU por 1 BRL')\naxes[1].boxplot(x, vert=False, patch_artist=True)\naxes[1].set_title('Boxplot de UYU/BRL')\naxes[1].set_xlabel('UYU por 1 BRL')\nplt.tight_layout()\nplt.show()

## 6. Retornos y autocorrelacion\n\nComo es una serie temporal financiera, no se debe asumir independencia automaticamente.

In [ ]:
ret = df['log_retorno_uyu_por_brl'].dropna()\n\nplt.figure(figsize=(10, 4))\nplt.hist(ret, bins=60, edgecolor='white')\nplt.title('Distribucion de log-retornos UYU/BRL')\nplt.xlabel('Log-retorno diario')\nplt.ylabel('Frecuencia')\nplt.show()\n\ndef autocorr(values, nlags=20):\n    values = np.asarray(values) - np.mean(values)\n    denom = np.dot(values, values)\n    return [1.0] + [float(np.dot(values[:-lag], values[lag:]) / denom) for lag in range(1, nlags + 1)]\n\nacf_values = autocorr(ret, 20)\npd.DataFrame({'lag': range(21), 'autocorrelacion': acf_values}).head(10)

## 7. Ajuste de distribuciones e intervalos\n\nComparacion simple de distribuciones candidatas para el nivel de la cotizacion e intervalos clasicos para la media.

In [ ]:
fits = []\nfor name, dist, params in [\n    ('normal', stats.norm, stats.norm.fit(x)),\n    ('lognormal', stats.lognorm, stats.lognorm.fit(x, floc=0)),\n    ('gamma', stats.gamma, stats.gamma.fit(x, floc=0)),\n]:\n    ks = stats.kstest(x, dist.cdf, args=params)\n    fits.append({'distribucion': name, 'ks_estadistico': ks.statistic, 'p_value': ks.pvalue})\npd.DataFrame(fits).sort_values('ks_estadistico')

In [ ]:
rows = []\nn = len(x)\nmean = x.mean()\nse = x.std(ddof=1) / np.sqrt(n)\nfor level in [0.90, 0.95, 0.99]:\n    alpha = 1 - level\n    crit = stats.t.ppf(1 - alpha / 2, df=n - 1)\n    rows.append({'nivel_confianza': level, 'media': mean, 'limite_inferior': mean - crit * se, 'limite_superior': mean + crit * se})\npd.DataFrame(rows)

## 8. Nota metodologica\n\nUna correlacion, autocorrelacion o relacion algebraica no demuestra causalidad economica. Ademas, como los datos forman una serie temporal, los intervalos clasicos pueden ser optimistas si hay dependencia temporal. Para una version posterior se puede implementar bootstrap por bloques o modelos especificos de series temporales.